In [0]:
%run ../config/config

In [0]:
# Imports
import sys
sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger
import time
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

In [0]:
# Logger load model
logger = MLOpsLogger(
    name='01_load_model',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '01_load_model'
    }
)

dbutils.jobs.taskValues.set(key="environment", value=env)

In [0]:
try:
    logger.log_stage_start(
        'load_model',
        mes_vta=mes_vta,
        environment=env,
        model_table=model_table
    )

    start_time = time.time()

    logger.info("="*60)
    logger.info("VERIFICACIÓN DE MODELO(S) EN CATÁLOGO")
    logger.info("="*60)

    # ============================================================
    # Normalizar model_table a lista de rutas completas
    # ============================================================
    if isinstance(model_table, str):
        # Si es string, es la ruta completa de una sola tabla
        model_tables = [model_table]
    elif isinstance(model_table, list):
        # Si es lista, contiene solo los nombres de las tablas
        # Construir la ruta completa usando catalog_name y SCHEMA_MODELS
        model_tables = [f"{catalog_name}.{SCHEMA_MODELS}.{table_name}" for table_name in model_table]
    else:
        raise ValueError(f"model_table debe ser string o lista, recibido: {type(model_table)}")

    logger.info(f"Tablas a verificar: {len(model_tables)}")
    for idx, table in enumerate(model_tables, 1):
        logger.info(f"  {idx}. {table}")

    # ============================================================
    # Obtener sesión de Spark
    # ============================================================
    spark = SparkSession.builder.getOrCreate()

    # Variables para acumular resultados
    all_results = []
    total_columns_all_tables = 0
    total_records_all_tables = 0
    all_validations_passed = True

    # ============================================================
    # Iterar sobre cada tabla
    # ============================================================
    for table_idx, current_table in enumerate(model_tables, 1):
        logger.info("\n" + "="*60)
        logger.info(f"VERIFICANDO TABLA {table_idx}/{len(model_tables)}")
        logger.info("="*60)
        logger.info(f"Tabla: {current_table}")

        query_start = time.time()

        try:
            # Consulta SQL directa
            df = spark.sql(f"SELECT * FROM {current_table} LIMIT 10")

            query_duration = time.time() - query_start

            logger.info(f"✓ Consulta ejecutada exitosamente en {query_duration:.2f}s")

            # Obtener información básica del resultado
            record_count = df.count()
            column_count = len(df.columns)
            column_names = df.columns

            logger.info(f"\nRESULTADOS DE LA CONSULTA:")
            logger.info(f"  Registros obtenidos (muestra): {record_count}")
            logger.info(f"  Número de columnas: {column_count}")
            logger.info(f"  Columnas: {', '.join(column_names[:10])}{'...' if column_count > 10 else ''}")

            logger.log_performance(
                operation='query_model_table',
                duration=query_duration,
                records_retrieved=record_count,
                columns_count=column_count,
                table_name=current_table
            )

            # ============================================================
            # Validaciones adicionales
            # ============================================================
            logger.info(f"\nVERIFICANDO ESTRUCTURA DEL MODELO:")

            # Verificar columnas críticas del modelo
            required_columns = []
            missing_columns = [col for col in required_columns if col not in column_names]

            if missing_columns:
                logger.warning(
                    "Algunas columnas esperadas del modelo no fueron encontradas",
                    table=current_table,
                    missing_columns=missing_columns,
                    expected_columns=required_columns
                )
                logger.info(f" ⚠️ Columnas faltantes: {', '.join(missing_columns)}")
            else:
                logger.info(" ✓ Todas las columnas críticas del modelo están presentes")

            # Obtener el count total de la tabla
            logger.info(f"\nObteniendo count total de la tabla...")
            count_start = time.time()

            try:
                total_records = spark.sql(f"SELECT COUNT(*) as total FROM {current_table}").collect()[0]['total']
                count_duration = time.time() - count_start

                logger.info(f"  Total de registros: {total_records:,}")
                logger.info(f"  Tiempo de count: {count_duration:.2f}s")

                if total_records == 0:
                    logger.warning(
                        "La tabla del modelo está vacía",
                        table=current_table
                    )
                    all_validations_passed = False

                logger.log_performance(
                    operation='count_model_table',
                    duration=count_duration,
                    total_records=total_records,
                    table_name=current_table
                )

            except Exception as e:
                logger.warning(
                    "No se pudo obtener el count total",
                    table=current_table,
                    error_message=str(e)
                )
                total_records = -1
                all_validations_passed = False

            # Mostrar muestra de datos
            logger.info(f"\nMostrando muestra de datos de {current_table}:")
            display(df)

            # Guardar resultados de esta tabla
            table_result = {
                'table': current_table,
                'validated': True,
                'column_count': column_count,
                'sample_records': record_count,
                'total_records': total_records if total_records >= 0 else None,
                'missing_columns': missing_columns,
                'query_duration': query_duration
            }
            all_results.append(table_result)

            # Acumular métricas
            total_columns_all_tables += column_count
            if total_records >= 0:
                total_records_all_tables += total_records

            # Métricas de calidad por tabla
            logger.log_data_quality(
                dataset_name=f'model_table_{table_idx}',
                total_records=total_records if total_records >= 0 else 0,
                valid_records=record_count,
                invalid_records=0,
                columns_count=column_count,
                table_accessible=True,
                table_name=current_table
            )

            logger.info(f"\n✓ Tabla {current_table} verificada exitosamente")

        except AnalysisException as ae:
            logger.error(
                "La tabla del modelo NO existe o no es accesible",
                table=current_table,
                error_message=str(ae)
            )
            table_result = {
                'table': current_table,
                'validated': False,
                'error': str(ae)
            }
            all_results.append(table_result)
            all_validations_passed = False
            logger.warning(f"X Tabla {current_table} falló la verificación")

        except Exception as e:
            logger.error(
                "Error inesperado al consultar la tabla",
                table=current_table,
                error_type=type(e).__name__,
                error_message=str(e)
            )
            table_result = {
                'table': current_table,
                'validated': False,
                'error': str(e)
            }
            all_results.append(table_result)
            all_validations_passed = False
            logger.warning(f"X Tabla {current_table} falló la verificación")

    # ============================================================
    # RESUMEN CONSOLIDADO
    # ============================================================
    duration = time.time() - start_time

    logger.info("\n" + "="*60)
    logger.info("RESUMEN DE VERIFICACIÓN DE TODAS LAS TABLAS")
    logger.info("="*60)

    tables_validated = sum(1 for r in all_results if r.get('validated', False))
    tables_failed = len(all_results) - tables_validated

    logger.info(f"Total de tablas procesadas: {len(model_tables)}")
    logger.info(f"Tablas validadas exitosamente: {tables_validated}")
    if tables_failed > 0:
        logger.info(f"Tablas con errores: {tables_failed}")

    # Detalle de cada tabla
    for idx, result in enumerate(all_results, 1):
        status = "✓" if result.get('validated', False) else "X"
        logger.info(f"\n{status} Tabla {idx}: {result['table']}")
        if result.get('validated', False):
            logger.info(f"  Columnas: {result['column_count']}")
            if result.get('total_records') is not None:
                logger.info(f"  Registros: {result['total_records']:,}")
            if result.get('missing_columns'):
                logger.info(f"  Columnas faltantes: {', '.join(result['missing_columns'])}")
        else:
            logger.info(f"  Error: {result.get('error', 'Desconocido')}")

    if total_records_all_tables > 0:
        logger.info(f"\nTotal de registros (todas las tablas): {total_records_all_tables:,}")

    logger.info(f"\nTiempo total de verificación: {duration:.2f}s")
    logger.info("="*60)

    # Guardar información en task values
    dbutils.jobs.taskValues.set(key="model_table_validated", value=all_validations_passed)
    dbutils.jobs.taskValues.set(key="tables_validated_count", value=tables_validated)
    dbutils.jobs.taskValues.set(key="tables_failed_count", value=tables_failed)
    dbutils.jobs.taskValues.set(key="total_records_all_tables", value=total_records_all_tables)

    # Guardar resultados detallados como JSON
    import json
    dbutils.jobs.taskValues.set(key="validation_results", value=json.dumps(all_results))

    logger.log_stage_end(
        'load_model',
        duration=duration,
        model_loaded=all_validations_passed,
        model_validated=all_validations_passed,
        tables_processed=len(model_tables),
        tables_validated=tables_validated,
        tables_failed=tables_failed,
        total_records=total_records_all_tables
    )

    logger.log_performance(
        operation='load_model_complete',
        duration=duration,
        model_tables=model_tables,
        tables_validated=tables_validated,
        total_records=total_records_all_tables
    )

    logger.metric(
        "Verificación de modelos completada",
        event_type='models_loaded',
        tables_count=len(model_tables),
        tables_validated=tables_validated,
        tables_failed=tables_failed,
        all_validated=all_validations_passed
    )

    # Lanzar excepción si alguna tabla falló
    if not all_validations_passed:
        failed_tables = [r['table'] for r in all_results if not r.get('validated', False)]
        raise ValueError(
            f"Verificación falló para {tables_failed} tabla(s): {', '.join(failed_tables)}. "
            f"Verifique que los modelos han sido publicados correctamente desde SAS Viya Cloud."
        )

except Exception as e:
    logger.log_exception(
        operation='load_model',
        exception=e,
        should_raise=True,
        mes_vta=mes_vta,
        environment=env,
        model_table=model_table
    )

    # Marcar validación como fallida
    dbutils.jobs.taskValues.set(key="model_table_validated", value=False)

finally:
    # Cerrar logger y hacer flush final
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()